# 🚀 Notebook do Professor (Demo) — Aula 07: RAG avançado — chunking estratégico, reranking e RAGAS

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 07/14 — Módulo 2: RAG · 🏁 Entrega CKP02**  
**⏱️ 1h40min**  
**📊 faithfulness · answer_relevancy**  
**🏁 CKP02 entrega**  

---

## 🎯 Objetivo da aula

Sair da fase "funciona" para a fase "funciona bem". Medir objetivamente a qualidade do RAG com RAGAS, experimentar duas estratégias de chunking e documentar qual configuração entrega melhores resultados para o domínio do grupo. Isso é o CKP02.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz as soluções resolvidas dos exercícios da aula.

---

# 🔬 Código da aula — slide a slide

### Slide 06 — Três estratégias de chunking — quando usar cada uma

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

# SemanticChunker usa o modelo de embedding para decidir onde quebrar
semantic_splitter = SemanticChunker(
    embeddings,                         # mesmo OllamaEmbeddings da Aula 06
    breakpoint_threshold_type="percentile",  # quebra nos 95% de maior divergência
)
chunks_sem = semantic_splitter.split_documents(paginas)
print(f"Chunks semânticos: {len(chunks_sem)}")
print(f"Tamanho médio: {sum(len(c.page_content) for c in chunks_sem)/len(chunks_sem):.0f} chars")

### Slide 07 — Parent-Document Retriever — o melhor dos dois mundos

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

# Splitter pequeno — para indexação e busca (precisão)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

# Splitter grande — para o contexto enviado ao LLM (riqueza)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)

# Armazenamento para os chunks pai (não vai no ChromaDB)
store = InMemoryStore()

retriever_pd = ParentDocumentRetriever(
    vectorstore=db,             # ChromaDB com chunks de 200 chars
    docstore=store,            # store com chunks de 1000 chars
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
retriever_pd.add_documents(paginas)

# Como funciona:
# 1. Query → ChromaDB busca nos chunks de 200 chars (alta precisão)
# 2. Encontra chunk filho → recupera o chunk PAI de 1000 chars do store
# 3. Retorna o chunk de 1000 chars para o LLM (contexto rico)
docs = retriever_pd.invoke("prazo de garantia")
print(f"Docs retornados: {len(docs)}")
print(f"Tamanho do 1º: {len(docs[0].page_content)} chars")  # → ~1000

### Slide 09 — Reranking — o segundo filtro de relevância

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker

# cross-encoder leve, gratuito (roda em CPU no Colab)
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
compressor    = CrossEncoderReranker(model=cross_encoder, top_n=3)

# Combina retriever (busca ampla, k=10) + reranker (filtra para top_n=3)
retriever_rerank = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=db.as_retriever(search_kwargs={"k":10}),  # busca ampla
)
# Trocar o retriever na chain RAG é tudo que precisa mudar

### Slide 12 — RAGAS — implementação simplificada para o CKP02

In [ ]:
!pip install ragas -q

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# Configurar o RAGAS para usar o Ollama (sem custo)
ragas_llm  = LangchainLLMWrapper(ChatOllama(model="gpt-oss:120b", temperature=0))
ragas_embs = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))

faithfulness.llm       = ragas_llm
answer_relevancy.llm   = ragas_llm
answer_relevancy.embeddings = ragas_embs

# Dataset de avaliação — 5 pares preparados antes da aula
dados_avaliacao = {
    "question": [
        "Qual é o prazo de garantia do produto?",
        "Como acionar o suporte técnico?",
    ],
    "answer": [
        chain_rag.invoke("Qual é o prazo de garantia do produto?"),
        chain_rag.invoke("Como acionar o suporte técnico?"),
    ],
    "contexts": [
        [d.page_content for d in retriever.invoke("garantia")],
        [d.page_content for d in retriever.invoke("suporte técnico")],
    ],
}

resultado = evaluate(
    Dataset.from_dict(dados_avaliacao),
    metrics=[faithfulness, answer_relevancy],
)
print(resultado.to_pandas()[["question", "faithfulness", "answer_relevancy"]])

### Slide 14 — Fallback — RAGAS manual se o pacote tiver problema de compatibilidade

In [ ]:
# Implementação manual de faithfulness (LLM-as-judge)
PROMPT_JUIZ = """<tarefa>
Você é um avaliador de qualidade de sistemas RAG.
Avalie se a RESPOSTA está fundamentada no CONTEXTO.
</tarefa>

<contexto>{contexto}</contexto>
<resposta>{resposta}</resposta>

Responda APENAS com um número de 0 a 1:
- 1.0: toda a resposta está no contexto
- 0.5: resposta parcialmente no contexto
- 0.0: resposta inventa informações não presentes no contexto

Resposta (só o número):"""

def faithfulness_manual(pergunta, resposta, contexto) -> float:
    juiz = ChatOllama(model="gpt-oss:120b", temperature=0)
    prompt = ChatPromptTemplate.from_template(PROMPT_JUIZ)
    chain_juiz = prompt | juiz | StrOutputParser()
    resultado = chain_juiz.invoke({
        "contexto": contexto, "resposta": resposta
    })
    try:
        return float(resultado.strip())
    except:
        return -1.0  # score inválido — revisar resposta do modelo

# Aplicar a todas as perguntas
scores = []
for q in PERGUNTAS:
    resp = chain_rag.invoke(q)
    ctx  = "\n".join(d.page_content for d in retriever.invoke(q))
    scores.append(faithfulness_manual(q, resp, ctx))
print(f"Faithfulness médio: {sum(scores)/len(scores):.3f}")

### Slide 16 — Código da demo — comparar chunk_size com RAGAS

In [ ]:
def avaliar_chunking(paginas, chunk_size: int, perguntas: list) -> dict:
    """Cria pipeline RAG com chunk_size dado e retorna scores RAGAS."""
    chunks    = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_size//8).split_documents(paginas)
    retriever = Chroma.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k":3})
    chain     = montar_chain_rag(retriever)

    dados = {"question":[], "answer":[], "contexts":[]}
    for q in perguntas:
        dados["question"].append(q)
        dados["answer"].append(chain.invoke(q))
        dados["contexts"].append([d.page_content for d in retriever.invoke(q)])

    res = evaluate(Dataset.from_dict(dados), metrics=[faithfulness, answer_relevancy])
    return {"chunk_size": chunk_size,
            "faithfulness":     res["faithfulness"],
            "answer_relevancy": res["answer_relevancy"]}

for cs in [256, 512, 1024]:
    r = avaliar_chunking(paginas, cs, PERGUNTAS_AVALIACAO)
    print(f"chunk={r['chunk_size']:4d} | faith={r['faithfulness']:.3f} | rel={r['answer_relevancy']:.3f}")

### Slide 17 — Python novo desta aula

In [ ]:
# 1. Divisão inteira com // — chunk_overlap como fração do chunk_size
chunk_size    = 512
chunk_overlap = chunk_size // 8  # → 64 (divisão inteira, sem float)

# 2. Loop com tupla de 3 elementos — (nome, chain, retriever)
for nome, chain, retr in [
    ("A", chain_a, retriever_a),
    ("B", chain_b, retriever_b),
]:
    print(f"Testando {nome}")

# 3. try / except com ValueError para float parsing
try:
    score = float(texto.strip())  # LLM pode retornar "0.87" ou texto extra
except ValueError:
    score = -1.0               # sentinela: score inválido

# 4. List comprehension aninhado — contexts para o RAGAS
contexts = [
    [d.page_content for d in retriever.invoke(q)]
    for q in perguntas
]  # lista de listas: [["chunk1", "chunk2", ...], [...], ...]

# 5. sum() / len() para média simples
media = sum(scores) / len(scores)  # sem precisar importar statistics

# 6. Dataset.from_dict() — criar dataset HuggingFace de um dict
from datasets import Dataset
ds = Dataset.from_dict({"col_a": [1,2], "col_b": ["x","y"]})

---

## 🏋️ Exercícios Resolvidos — versão professor (executar no Colab)

As quatro soluções prontas dos exercícios de fixação do notebook do aluno — rode em sala, uma a uma.


### Exercício 1 — Chunking semântico

**O que a solução demonstra:** o `SemanticChunker` em execução — `breakpoint_threshold_type="percentile"` e `breakpoint_threshold_amount=95` — comparado ao Recursive-512 em contagem e tamanho médio.

**Pontos a destacar na execução:** abra um chunk e mostre que o corte respeita a fronteira de sentido — o embedding decide onde quebrar. Conecte com o trade-off: preciso no match × rico no contexto.


In [ ]:
# ── Solução do Exercício 1 — chunking semântico ──
!pip install langchain langchain-community langchain-ollama chromadb langchain-experimental langchain-text-splitters -q

import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Corpus de exemplo — com PDFs reais: paginas = PyMuPDFLoader(pdf).load()
paginas = [
    Document(page_content="Regulamento do produto — Seção 1: Garantia. Art. 1º A garantia cobre 12 meses contra defeitos de fabricação, contados da data da nota fiscal. Art. 2º A garantia não cobre danos por mau uso, queda ou contato com líquidos. Parágrafo único. O conserto em garantia é executado por assistência autorizada, sem custo para o consumidor.", metadata={"source": "regulamento_demo.pdf", "page": 0}),
    Document(page_content="Seção 2: Suporte. Art. 3º O suporte técnico atende em dias úteis, das 9h às 18h, pelo portal e pelo chat. Art. 4º O primeiro retorno ocorre em até 48 horas úteis após a abertura do chamado. Art. 5º Casos de urgência têm fila prioritária no atendimento.", metadata={"source": "regulamento_demo.pdf", "page": 1}),
    Document(page_content="Seção 3: Entrega e devolução. Art. 6º O prazo de entrega é de 5 dias úteis para capitais e 10 dias úteis para o interior. Art. 7º A devolução é aceita em até 7 dias corridos, com a embalagem original. Art. 8º O reembolso ocorre em 10 dias após o recebimento do produto.", metadata={"source": "regulamento_demo.pdf", "page": 2}),
]

chunks_rec = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64).split_documents(paginas)
print(f"Recursive-512: {len(chunks_rec)} chunks · média {sum(len(c.page_content) for c in chunks_rec)/len(chunks_rec):.0f} chars")

# Lacunas 1 e 2 resolvidas — o corte por percentile usa o próprio embedding
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95,
)
chunks_sem = semantic_splitter.split_documents(paginas)
print(f"SemanticChunker: {len(chunks_sem)} chunks · média {sum(len(c.page_content) for c in chunks_sem)/len(chunks_sem):.0f} chars")
print(f"\nExemplo de chunk semântico:\n{chunks_sem[0].page_content[:200]}")
# Destaque em sala: o corte respeita fronteiras de sentido — o embedding decide onde quebrar.


### Exercício 2 — Reranking com cross-encoder

**O que a solução demonstra:** os dois retrievers sobre o mesmo vector store — o simples (k=3) e o com reranking (busca ampla k=10 + `CrossEncoderReranker` top_n=3) — rodando as mesmas perguntas com as ordens lado a lado no console.

**Pontos a destacar na execução:** o cross-encoder costuma reordenar o pódio e derrubar chunks com boa distância cosseno mas sem resposta direta à pergunta. Explique o custo: ele lê o par (query, chunk) completo — mais preciso, por isso só nos 10 candidatos.


In [ ]:
# ── Solução do Exercício 2 (parte 1) — os dois retrievers ──
!pip install langchain langchain-community langchain-ollama chromadb langchain-text-splitters sentence-transformers -q

import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.documents import Document

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Corpus com distratores — chunks parecidos na superfície, úteis ou não para a query
documentos = [
    Document(page_content="A garantia do produto cobre 12 meses contra defeitos de fabricação a partir da data da nota fiscal."),
    Document(page_content="O suporte técnico atende em dias úteis, das 9h às 18h, pelo portal e pelo chat, com retorno em até 48 horas úteis."),
    Document(page_content="A embalagem do produto é reciclável e o manual impresso usa papel certificado."),
    Document(page_content="O prazo de entrega para capitais é de 5 dias úteis após a confirmação do pagamento."),
    Document(page_content="A empresa foi criada para vender produtos de qualidade com garantia estendida opcional."),
    Document(page_content="A garantia estendida pode ser adquirida no ato da compra e adiciona 12 meses de cobertura."),
]
chunks = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50).split_documents(documentos)
db = Chroma.from_documents(chunks, embeddings, collection_name="ex02_rerank")

retriever_simples = db.as_retriever(search_kwargs={"k": 3})   # só distância cosseno

# cross-encoder leve e gratuito (roda em CPU no Colab) — Slide 09
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)
retriever_rerank = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=db.as_retriever(search_kwargs={"k": 10}),  # busca ampla → filtra para 3
)
print("Retrievers prontos: simples (k=3) e com reranking (k=10 → top_n=3).")


In [ ]:
# ── Solução do Exercício 2 (parte 2) — muda o pódio? ──
for q in [
    "Qual é o prazo de garantia?",
    "Como acionar o suporte técnico?",
    "A embalagem do produto é reciclável?",
]:
    print(f"\nQuery: {q}")
    print("── top-3 SEM reranking (ordem da distância cosseno):")
    for d in retriever_simples.invoke(q):
        print(f"   {d.page_content[:70]}")
    print("── top-3 COM reranking (cross-encoder reordena e filtra):")
    for d in retriever_rerank.invoke(q):
        print(f"   {d.page_content[:70]}")
# Observação em sala: o cross-encoder costuma reordenar o pódio e derrubar
# chunks com boa distância cosseno mas sem resposta direta à pergunta.


### Exercício 3 — Avaliação com RAGAS

**O que a solução demonstra:** o RAGAS configurado com o Ollama (sem custo) avaliando a estratégia A (Recursive-512) nas perguntas do domínio — `faithfulness` e `answer_relevancy` por pergunta, ordenadas do pior para o melhor.

**Pontos a destacar na execução:** as métricas preenchidas são `faithfulness` e `answer_relevancy` (Slide 12) e o juiz roda com `temperature=0`. Scores baixos vão para o diagnóstico do Exercício 4.


In [ ]:
# ── Solução do Exercício 3 (parte 1) — estratégia A + RAGAS configurado ──
!pip install langchain langchain-community langchain-ollama chromadb ragas datasets langchain-text-splitters -q

import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

from langchain_community.document_loaders import PyMuPDFLoader   # PDFs reais do grupo
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# RAGAS com Ollama — sem custo (Slide 12)
ragas_llm  = LangchainLLMWrapper(ChatOllama(model="gpt-oss:120b", temperature=0))
ragas_embs = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))
faithfulness.llm            = ragas_llm
answer_relevancy.llm        = ragas_llm
answer_relevancy.embeddings = ragas_embs

PROMPT_RAG = """<persona>
Você é um assistente especializado em responder perguntas
com base EXCLUSIVAMENTE nos documentos fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do contexto abaixo.
- Sempre cite a fonte: (fonte: {nome_doc}, página X).
- Se a resposta não estiver no contexto, diga:
  "Não encontrei essa informação nos documentos fornecidos."
- Nunca invente, extrapole ou use conhecimento externo.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""

def montar_chain_rag(retriever, nome_doc):
    def formatar_contexto(docs) -> str:
        return "\n\n---\n\n".join(
            f"[{d.metadata.get('source','?')}, pág. {d.metadata.get('page',0)+1}]\n{d.page_content}"
            for d in docs
        )
    prompt = ChatPromptTemplate.from_template(PROMPT_RAG)
    return (
        {"contexto": retriever | RunnableLambda(formatar_contexto),
         "pergunta": RunnablePassthrough(),
         "nome_doc": RunnableLambda(lambda _: nome_doc)}
        | prompt | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
    )

# Estratégia A do andaime — Recursive-512 (com PDFs reais: paginas = PyMuPDFLoader(pdf).load())
paginas = [
    Document(page_content="Regulamento do produto — Seção 1: Garantia. Art. 1º A garantia cobre 12 meses contra defeitos de fabricação, contados da data da nota fiscal. Art. 2º A garantia não cobre danos por mau uso, queda ou contato com líquidos. Parágrafo único. O conserto em garantia é executado por assistência autorizada, sem custo para o consumidor.", metadata={"source": "regulamento_demo.pdf", "page": 0}),
    Document(page_content="Seção 2: Suporte. Art. 3º O suporte técnico atende em dias úteis, das 9h às 18h, pelo portal e pelo chat. Art. 4º O primeiro retorno ocorre em até 48 horas úteis após a abertura do chamado. Art. 5º Casos de urgência têm fila prioritária no atendimento.", metadata={"source": "regulamento_demo.pdf", "page": 1}),
    Document(page_content="Seção 3: Entrega e devolução. Art. 6º O prazo de entrega é de 5 dias úteis para capitais e 10 dias úteis para o interior. Art. 7º A devolução é aceita em até 7 dias corridos, com a embalagem original. Art. 8º O reembolso ocorre em 10 dias após o recebimento do produto.", metadata={"source": "regulamento_demo.pdf", "page": 2}),
]
chunks_a    = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64).split_documents(paginas)
db_a        = Chroma.from_documents(chunks_a, embeddings, collection_name="ex03_ragas")
retriever_a = db_a.as_retriever(search_kwargs={"k": 3})
chain_a     = montar_chain_rag(retriever_a, "regulamento_demo.pdf")

PERGUNTAS = [
    "Qual é o prazo de garantia do produto?",
    "Como acionar o suporte técnico?",
    "Qual é o prazo de entrega para capitais?",
    "Em quantos dias a devolução é aceita?",
    "Qual é o telefone da central de atendimento?",   # NÃO está nos documentos
]
print(f"Pipeline A pronto · chunks={len(chunks_a)} · perguntas={len(PERGUNTAS)}")


In [ ]:
# ── Solução do Exercício 3 (parte 2) — a avaliação ──
dados = {
    "question": PERGUNTAS,
    "answer":   [chain_a.invoke(q) for q in PERGUNTAS],
    "contexts": [[d.page_content for d in retriever_a.invoke(q)] for q in PERGUNTAS],
}
res = evaluate(Dataset.from_dict(dados), metrics=[faithfulness, answer_relevancy])
print(res.to_pandas()[["question", "faithfulness", "answer_relevancy"]].sort_values("faithfulness"))
# Métricas preenchidas: faithfulness e answer_relevancy (Slide 12) — juiz com temperature=0.


### Exercício 4 — Diagnóstico de faithfulness

**O que a solução demonstra:** o diagnóstico completo do pior caso — ordenação por `faithfulness` crescente e inspeção dos chunks que chegam ao contexto da pergunta crítica.

**Pontos a destacar na execução:** leitura da falha — trecho PRESENTE no contexto com resposta errada = problema de prompt/LLM (reforçar grounding, `temperature=0`); trecho AUSENTE = problema de retrieval (reduzir `chunk_size`/overlap ou aumentar `k`). A correção proposta aqui é insumo direto do CKP02.


In [ ]:
# ── Solução do Exercício 4 (parte 1) — estratégia A + RAGAS configurado ──
!pip install langchain langchain-community langchain-ollama chromadb ragas datasets langchain-text-splitters -q

import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

from langchain_community.document_loaders import PyMuPDFLoader   # PDFs reais do grupo
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# RAGAS com Ollama — sem custo (Slide 12)
ragas_llm  = LangchainLLMWrapper(ChatOllama(model="gpt-oss:120b", temperature=0))
ragas_embs = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))
faithfulness.llm            = ragas_llm
answer_relevancy.llm        = ragas_llm
answer_relevancy.embeddings = ragas_embs

PROMPT_RAG = """<persona>
Você é um assistente especializado em responder perguntas
com base EXCLUSIVAMENTE nos documentos fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do contexto abaixo.
- Sempre cite a fonte: (fonte: {nome_doc}, página X).
- Se a resposta não estiver no contexto, diga:
  "Não encontrei essa informação nos documentos fornecidos."
- Nunca invente, extrapole ou use conhecimento externo.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""

def montar_chain_rag(retriever, nome_doc):
    def formatar_contexto(docs) -> str:
        return "\n\n---\n\n".join(
            f"[{d.metadata.get('source','?')}, pág. {d.metadata.get('page',0)+1}]\n{d.page_content}"
            for d in docs
        )
    prompt = ChatPromptTemplate.from_template(PROMPT_RAG)
    return (
        {"contexto": retriever | RunnableLambda(formatar_contexto),
         "pergunta": RunnablePassthrough(),
         "nome_doc": RunnableLambda(lambda _: nome_doc)}
        | prompt | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
    )

# Estratégia A do andaime — Recursive-512 (com PDFs reais: paginas = PyMuPDFLoader(pdf).load())
paginas = [
    Document(page_content="Regulamento do produto — Seção 1: Garantia. Art. 1º A garantia cobre 12 meses contra defeitos de fabricação, contados da data da nota fiscal. Art. 2º A garantia não cobre danos por mau uso, queda ou contato com líquidos. Parágrafo único. O conserto em garantia é executado por assistência autorizada, sem custo para o consumidor.", metadata={"source": "regulamento_demo.pdf", "page": 0}),
    Document(page_content="Seção 2: Suporte. Art. 3º O suporte técnico atende em dias úteis, das 9h às 18h, pelo portal e pelo chat. Art. 4º O primeiro retorno ocorre em até 48 horas úteis após a abertura do chamado. Art. 5º Casos de urgência têm fila prioritária no atendimento.", metadata={"source": "regulamento_demo.pdf", "page": 1}),
    Document(page_content="Seção 3: Entrega e devolução. Art. 6º O prazo de entrega é de 5 dias úteis para capitais e 10 dias úteis para o interior. Art. 7º A devolução é aceita em até 7 dias corridos, com a embalagem original. Art. 8º O reembolso ocorre em 10 dias após o recebimento do produto.", metadata={"source": "regulamento_demo.pdf", "page": 2}),
]
chunks_a    = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64).split_documents(paginas)
db_a        = Chroma.from_documents(chunks_a, embeddings, collection_name="ex04_diag")
retriever_a = db_a.as_retriever(search_kwargs={"k": 3})
chain_a     = montar_chain_rag(retriever_a, "regulamento_demo.pdf")

PERGUNTAS = [
    "Qual é o prazo de garantia do produto?",
    "Como acionar o suporte técnico?",
    "Qual é o prazo de entrega para capitais?",
    "Em quantos dias a devolução é aceita?",
    "Qual é o telefone da central de atendimento?",   # NÃO está nos documentos
]
print(f"Pipeline A pronto · chunks={len(chunks_a)} · perguntas={len(PERGUNTAS)}")


In [ ]:
# ── Solução do Exercício 4 (parte 2) — avaliação RAGAS e diagnóstico da pior pergunta ──
dados_diag = {
    "question": PERGUNTAS,
    "answer":   [chain_a.invoke(q) for q in PERGUNTAS],
    "contexts": [[d.page_content for d in retriever_a.invoke(q)] for q in PERGUNTAS],
}
res_diag = evaluate(Dataset.from_dict(dados_diag), metrics=[faithfulness, answer_relevancy])
df = res_diag.to_pandas()
print(df[["question", "faithfulness", "answer_relevancy"]].sort_values("faithfulness"))

# Inspecionar os chunks da PIOR pergunta
pior_q = df.sort_values("faithfulness").iloc[0]["question"]
print(f"\nPergunta crítica: {pior_q}\n")
for i, d in enumerate(retriever_a.invoke(pior_q), 1):
    print(f"[chunk {i}] {d.page_content[:150]}")

# Classificação da falha — regra de leitura para a sala:
#   trecho PRESENTE no contexto + resposta errada → problema no prompt/LLM
#     → reforçar grounding no prompt e confirmar temperature=0
#   trecho AUSENTE do contexto → problema no retriever
#     → reduzir chunk_size/overlap ou aumentar k


## 📚 Referências da aula

- Paper Es, S. et al. — "RAGAS: Automated Evaluation of Retrieval Augmented Generation." EACL, 2024. O paper que define faithfulness e answer_relevancy. arxiv.org/abs/2309.15217
- Docs RAGAS — Documentação oficial: métricas, integração com Ollama, datasets. docs.ragas.io
- Docs LangChain — SemanticChunker e ParentDocumentRetriever. python.langchain.com/docs/how_to/semantic-chunker
- Modelo cross-encoder/ms-marco-MiniLM-L-6-v2 — Modelo de reranking leve (22M params). huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15 — Representações distribuídas: a base teórica dos embeddings que sustentam o chunking semântico.

---

**→ Próxima Aula — Aula 08 · 29/Set** — Interfaces com Gradio e Streamlit
  
RAG com URL pública. RunnableWithMessageHistory para memória entre turnos.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*